# Stance sequential stability example

This notebook uses the archived sequential Monte Carlo results to make paper-facing stability plots. The raw result files keep the historical estimator IDs (`spline`, `spline+`), while the plotting helpers display them as `OPAL` and `OPAL + tuning`.

In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import sys

import numpy as np
import pandas as pd

CWD = Path.cwd().resolve()
REPO_ROOT = CWD.parent if CWD.name in {"BRCA", "CheXpert", "Alphafold", "Stance"} else CWD
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from utils import EFFECTIVE_N_COL, HUMAN_N_COL
from plotting import (
    METHOD_LABELS,
    plot_coverage,
    plot_effective_sample_size,
    plot_sequential_effective_sample_size,
)


In [ ]:
ALPHA = 0.10
MAX_ITERATIONS = 50

EXAMPLE_DIR = REPO_ROOT / "Stance"
ARCHIVE_RESULTS_PATH = EXAMPLE_DIR / "archive" / "stance_sequential_results_old.csv"
RESULTS_DIR = EXAMPLE_DIR / "results"
PLOTS_DIR = EXAMPLE_DIR / "plots"

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
PLOTS_DIR.mkdir(parents=True, exist_ok=True)


## Load archived sequential results

The sequential archive stores one row per Monte Carlo trial, budget, and estimator. We copy the cleaned version into `Stance/results/` so subsequent plotting code has a stable input path.

In [ ]:
seq_results = pd.read_csv(ARCHIVE_RESULTS_PATH)
seq_results[HUMAN_N_COL] = pd.to_numeric(seq_results[HUMAN_N_COL])
seq_results[EFFECTIVE_N_COL] = pd.to_numeric(seq_results[EFFECTIVE_N_COL])
seq_results["coverage"] = seq_results["coverage"].astype(bool)
seq_results = seq_results.sort_values([HUMAN_N_COL, "num_trial", "estimator"]).reset_index(drop=True)

seq_results.to_csv(RESULTS_DIR / "Stance_sequential_results.csv", index=False)

max_budget = float(seq_results[HUMAN_N_COL].max())
n_total = int(round(max_budget / 0.5))
seq_results.head()


In [ ]:
sequential_summary = (
    seq_results.groupby([HUMAN_N_COL, "estimator"], observed=True)
    .agg(
        interval_width_mean=("interval width", "mean"),
        coverage=("coverage", "mean"),
        effective_n_mean=(EFFECTIVE_N_COL, "mean"),
        effective_n_sd=(EFFECTIVE_N_COL, "std"),
        trials=("num_trial", "nunique"),
    )
    .reset_index()
)
sequential_summary.to_csv(RESULTS_DIR / "Stance_sequential_summary.csv", index=False)
sequential_summary


## Stability at the largest sequential budget

The plot fixes the largest human-label budget and shows effective sample size across Monte Carlo trial index. This is the sequential stability view requested for the Stance example.

In [ ]:
plot_sequential_effective_sample_size(
    seq_results,
    path=PLOTS_DIR / "Stance_sequential_effective_sample_size_per_iteration.png",
    n_human=max_budget,
    max_iterations=MAX_ITERATIONS,
    title="Effective Sample Size per Iteration",
)


In [ ]:
stability_table = (
    seq_results[seq_results[HUMAN_N_COL].eq(max_budget)]
    .groupby("estimator", observed=True)[EFFECTIVE_N_COL]
    .agg(mean="mean", sd="std", minimum="min", maximum="max", trials="count")
    .rename(index=lambda method: METHOD_LABELS.get(method, method))
    .sort_values("mean", ascending=False)
)
stability_table


## Standard sequential budget plots

These mirror the batch notebook outputs: an ESS plot with mean +/- 1 SD bars, a no-error-bar ESS version, and empirical coverage by budget.

In [ ]:
plot_effective_sample_size(
    seq_results,
    path=PLOTS_DIR / "Stance_sequential_effective_sample_size.png",
    n_total=n_total,
    error_bars="sd",
)

plot_effective_sample_size(
    seq_results,
    path=PLOTS_DIR / "Stance_sequential_effective_sample_size_no_error_bars.png",
    n_total=n_total,
    error_bars="none",
)

plot_coverage(
    seq_results,
    alpha=ALPHA,
    path=PLOTS_DIR / "Stance_sequential_coverage.png",
    n_total=n_total,
)
